# Association Rule Mining: Apriori

Please run the scripts in the following order: 1.extracting_script -> 2.geo_handle -> 3.association_mining

## Import the libraries

In [20]:
import pandas as pd 
import numpy as np
import mlxtend
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import association_rules

## Import the dataset

In [10]:
fact_fatalities = pd.read_csv('fact_fatalities.csv')
date_dimension = pd.read_csv('date_dimension.csv')
gender_dimension = pd.read_csv('gender_dimension.csv')
holiday_dimension = pd.read_csv('holiday_dimension.csv')
involvement_dimension = pd.read_csv('involvement_dimension.csv')
location_dimension = pd.read_csv('location_dimension.csv')
road_dimension = pd.read_csv('road_dimension.csv')
road_user_dimension = pd.read_csv('road_user_dimension.csv')
time_dimension = pd.read_csv('time_dimension.csv')

Merging the fact table with dimensional tables to join the interesting features

In [11]:
temp_table = pd.merge(fact_fatalities, date_dimension, on='DateKey', how='left')
temp_table = pd.merge(temp_table, gender_dimension, on = 'GenderKey', how='left')
temp_table = pd.merge(temp_table, holiday_dimension, on = 'HolidayKey', how='left')
temp_table = pd.merge(temp_table, involvement_dimension, on = 'InvolvementKey', how='left')
temp_table = pd.merge(temp_table, location_dimension, on = 'LocationKey', how='left')
temp_table = pd.merge(temp_table, road_dimension, on = 'RoadKey', how='left')
temp_table = pd.merge(temp_table, road_user_dimension, on = 'RoadUserKey', how='left')
temp_table = pd.merge(temp_table, time_dimension, on = 'TimeKey', how='left')

In [12]:
temp_table.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 55183 entries, 0 to 55182
Data columns (total 28 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   fat_id                       55183 non-null  int64 
 1   CrashKey                     55183 non-null  int64 
 2   DateKey                      55183 non-null  int64 
 3   TimeKey                      55183 non-null  int64 
 4   HolidayKey                   55183 non-null  int64 
 5   RoadUserKey                  55183 non-null  int64 
 6   GenderKey                    55183 non-null  int64 
 7   RoadKey                      55183 non-null  int64 
 8   InvolvementKey               55183 non-null  int64 
 9   SpeedLimit                   55183 non-null  int64 
 10  Age                          55183 non-null  int64 
 11  CrashTypeKey                 55183 non-null  object
 12  LocationKey                  55183 non-null  int64 
 13  Dayweek                      55

Categorize Numerical features and Weekday to reduce the complexity

In [21]:
temp_table['AgeBand'] = pd.cut(temp_table['Age'],
                               bins=[0, 30, 39, 65, 150],
                               labels =['Young','Mature','MidAge','Older'],
                               right=False)
temp_table['SpeedBand'] = pd.cut(temp_table['SpeedLimit'],
                                 bins=[0,20, 50, 100, 300],
                                 labels = ['Under20','From20to50','From50to100','Over100'],
                                 right=False)
temp_table['TimeBand'] = pd.cut(pd.to_datetime(temp_table['Time'], format='%H:%M:%S', errors='coerce').dt.hour,
                                 bins=[0, 5, 12, 17, 21, 24],
                                 labels = ['Night','Morning','Afternoon','Evening','Night'],
                                 right=False,
                                 ordered=False)
temp_table['Weekend'] = np.where(temp_table['Dayweek'].isin(['Saturday','Sunday']), 'Weekend', 'Weekday')
#temp_table['HeavyTrafficInvolved'] = temp_table[['BusInvolvement', 'HeavyRigidTruckInvolvement', 'ArticulatedTruckInvolvement']].apply(lambda x: 'HeavyTraffic' if 'Yes' in x.values else 'NoHeavyInvolvement', axis=1)

mining_table = temp_table[['Weekend','Gender','SpeedBand','TimeBand','AgeBand','State','RoadUser']].copy()
mining_table.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 55183 entries, 0 to 55182
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype   
---  ------     --------------  -----   
 0   Weekend    55183 non-null  object  
 1   Gender     55183 non-null  object  
 2   SpeedBand  55183 non-null  category
 3   TimeBand   55183 non-null  category
 4   AgeBand    55183 non-null  category
 5   State      55183 non-null  object  
 6   RoadUser   55183 non-null  object  
dtypes: category(3), object(4)
memory usage: 1.8+ MB


In [14]:
mining_table.head(5)

,Weekend,Gender,SpeedBand,TimeBand,AgeBand,State,LGA,RoadUser
0,Weekday,Male,Over100,Night,Older,NSW,Wagga Wagga,Driver
1,Weekday,Female,From50to100,Morning,Young,NSW,Hawkesbury,Driver
2,Weekday,Female,From50to100,Morning,Mature,Tas,Northern Midlands,Driver
3,Weekday,Female,Over100,Morning,Mature,NSW,Armidale,Driver
4,Weekday,Female,Over100,Afternoon,MidAge,Qld,Lockyer Valley,Passenger


Null data checking

In [15]:
mining_table.isna().sum()

Weekend      0
Gender       0
SpeedBand    0
TimeBand     0
AgeBand      0
State        0
LGA          0
RoadUser     0
dtype: int64

In [ ]:
new_df = mining_table.astype(str) #Convert the data to string type
list = new_df.values.tolist()
te = TransactionEncoder()
array_te = te.fit(list).transform(list) #Fit the data

In [23]:
array_te

array([[False, False,  True, ...,  True, False, False],
       [False, False,  True, ...,  True, False,  True],
       [False, False,  True, ...,  True, False, False],
       ...,
       [False, False, False, ...,  True, False,  True],
       [False, False,  True, ...,  True, False,  True],
       [False, False, False, ...,  True, False,  True]])

In [24]:
te.columns_

['ACT',
 'Afternoon',
 'Driver',
 'Evening',
 'Female',
 'From20to50',
 'From50to100',
 'Male',
 'Mature',
 'MidAge',
 'Morning',
 'Motorcycle pillion passenger',
 'Motorcycle rider',
 'NSW',
 'NT',
 'Night',
 'Older',
 'Other/-9',
 'Over100',
 'Passenger',
 'Pedal cyclist',
 'Pedestrian',
 'Qld',
 'SA',
 'Tas',
 'Under20',
 'Vic',
 'WA',
 'Weekday',
 'Weekend',
 'Young']

In [31]:
arm_df = pd.DataFrame(array_te, columns=te.columns_)
arm_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 55183 entries, 0 to 55182
Data columns (total 31 columns):
 #   Column                        Non-Null Count  Dtype
---  ------                        --------------  -----
 0   ACT                           55183 non-null  bool 
 1   Afternoon                     55183 non-null  bool 
 2   Driver                        55183 non-null  bool 
 3   Evening                       55183 non-null  bool 
 4   Female                        55183 non-null  bool 
 5   From20to50                    55183 non-null  bool 
 6   From50to100                   55183 non-null  bool 
 7   Male                          55183 non-null  bool 
 8   Mature                        55183 non-null  bool 
 9   MidAge                        55183 non-null  bool 
 10  Morning                       55183 non-null  bool 
 11  Motorcycle pillion passenger  55183 non-null  bool 
 12  Motorcycle rider              55183 non-null  bool 
 13  NSW                           5

In [32]:
frequent_itemsets = apriori(arm_df, min_support = 0.05, use_colnames=True)
frequent_itemsets['length'] = frequent_itemsets['itemsets'].apply(lambda x: len(x))

In [37]:
rules_con = association_rules(frequent_itemsets, metric='confidence', min_threshold=0.5)
rules_arm = rules_con[['antecedents','consequents','support','confidence','lift']].copy()
rules_arm

,antecedents,consequents,support,confidence,lift
0,(Afternoon),(Male),0.185184,0.658017,0.915727
1,(Afternoon),(Over100),0.145842,0.518223,1.099254
2,(Afternoon),(Weekday),0.193755,0.688474,1.045030
3,(Driver),(Male),0.341681,0.752454,1.047151
4,(Mature),(Driver),0.071906,0.516667,1.137809
...,...,...,...,...,...
440,"(Weekend, Night, Male)",(Young),0.058061,0.642342,1.565858
441,"(Weekend, Night)","(Young, Male)",0.058061,0.522675,1.708190
442,"(Over100, Young, Weekday)",(Male),0.078104,0.691592,0.962452
443,"(Over100, Young, Male)",(Weekday),0.078104,0.577671,0.876842


In [26]:
def top_k_right(rule_arm, k, set, metric):
    # Top k Rule function to present the output on the metric of interest
    rule_consequent = rule_arm[rule_arm['consequents'].apply(lambda x: any(item in set for item in x))]
    rule_consequent = rule_consequent.sort_values(by=metric, ascending = False)
    return rule_consequent.head(k)
    

In [34]:
road_user_items = set(road_user_dimension['RoadUser']) #Create set of Road User value to filter on the RHS

In [ ]:
top_k_right(rules_arm, 5, road_user_items, 'lift') #Top 5 ranked by lift

,antecedents,consequents,support,confidence,lift
351,"(Over100, Night)","(Driver, Male)",0.054093,0.508691,1.488788
373,"(MidAge, Over100, Weekday)",(Driver),0.065002,0.661930,1.457710
327,"(MidAge, Over100, Male)",(Driver),0.066379,0.641057,1.411743
357,"(Over100, Weekday, Male)",(Driver),0.134770,0.634286,1.396831
334,"(Over100, Male, Morning)",(Driver),0.056684,0.633198,1.394436


In [ ]:
top_k_right(rules_arm, 5, road_user_items, 'confidence') #Top 5 ranked by confidence

,antecedents,consequents,support,confidence,lift
373,"(MidAge, Over100, Weekday)",(Driver),0.065002,0.661930,1.457710
327,"(MidAge, Over100, Male)",(Driver),0.066379,0.641057,1.411743
357,"(Over100, Weekday, Male)",(Driver),0.134770,0.634286,1.396831
334,"(Over100, Male, Morning)",(Driver),0.056684,0.633198,1.394436
348,"(Over100, Night, Male)",(Driver),0.054093,0.629083,1.385374
